### Name : Prajwal Mhase
### ROll NO : 93
### Div : B

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('Dry_Bean_Dataset.csv')

In [ ]:
df = df.dropna().drop_duplicates()


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Class"] = le.fit_transform(df["Class"])

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x="Class")
plt.title("Class Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(), cmap="coolwarm", center=0)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

print("After SMOTE :", X_train.shape, y_train.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
def create_model(optimizer):

    model = Sequential()

    model.add(Dense(64,
                    activation="relu",
                    kernel_regularizer=l2(0.001),
                    input_shape=(X_train.shape[1],)))

    model.add(BatchNormalization())

    model.add(Dropout(0.30))

    model.add(Dense(32,
                    activation="relu",
                    kernel_regularizer=l2(0.001)))

    model.add(BatchNormalization())

    model.add(Dropout(0.30))

    model.add(Dense(16,
                    activation="relu"))

    model.add(Dropout(0.20))

    # Change this according to the number of classes
    model.add(Dense(7,
                    activation="softmax"))

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
early = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
optimizers = {

    "Adam": Adam(0.001),

    "SGD": SGD(0.01),

    "RMSprop": RMSprop(0.001)

}

In [ ]:
summary = []

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report,
                             confusion_matrix)

for name, opt in optimizers.items():

    print("="*70)
    print("Training using", name)

    model = create_model(opt)

    history = model.fit(
        X_train,
        y_train,
        epochs=25,
        batch_size=256,
        validation_split=0.2,
        callbacks=[early]
    )

    # Accuracy Graph

    plt.figure(figsize=(6,4))

    plt.plot(history.history["accuracy"], label="Train")

    plt.plot(history.history["val_accuracy"], label="Validation")

    plt.title(f"{name} Accuracy")

    plt.legend()

    plt.grid(True)

    plt.show()

    # Loss Graph

    plt.figure(figsize=(6,4))

    plt.plot(history.history["loss"], label="Train")

    plt.plot(history.history["val_loss"], label="Validation")

    plt.title(f"{name} Loss")

    plt.legend()

    plt.grid(True)

    plt.show()

    # Prediction

    y_prob = model.predict(X_test)

    y_pred = np.argmax(y_prob, axis=1)
    
    # Metrics

    acc = accuracy_score(y_test, y_pred)

    pre = precision_score(y_test, y_pred, average="weighted")

    rec = recall_score(y_test, y_pred, average="weighted")

    f1 = f1_score(y_test, y_pred, average="weighted")

    print(classification_report(y_test, y_pred))

    # Confusion Matrix

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))

    sns.heatmap(cm,
                annot=True,
                fmt="d",
                cmap="Blues")

    plt.title(f"{name} Confusion Matrix")

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()

    summary.append([name, acc, pre, rec, f1])


In [ ]:
result = pd.DataFrame(
    summary,
    columns=[
        "Optimizer",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print(result)